# Phase 2 — The Notebook: Build & Evaluate the RAG Pipeline

### Project: RAG-Powered Document Assistant
**Domain**: FastAPI Framework Documentation (Tutorial and Advanced Sections)  
**Corpus Source**: Markdown documentation files in `data/raw/`  

---

This notebook implements the foundational data ingestion and chunking pipeline for an intelligent developer assistant specialized in the **FastAPI framework**.

In [1]:
import os
import sys
import re
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple

# Ensure project's virtualenv site-packages are accessible even if running under base/global kernel
for candidate in [
    Path("rag-assistant-project/.venv/lib/python3.13/site-packages"),
    Path(".venv/lib/python3.13/site-packages"),
    Path("../rag-assistant-project/.venv/lib/python3.13/site-packages"),
    Path("../.venv/lib/python3.13/site-packages"),
    Path("/Users/noura/.rag-venv/lib/python3.13/site-packages"),
]:
    resolved = candidate.resolve()
    if resolved.exists() and str(resolved) not in sys.path:
        sys.path.insert(0, str(resolved))

import pandas as pd
import numpy as np

# Locate data/raw directory flexibly whether running from repo root or notebooks/ dir
def resolve_data_dir() -> Path:
    candidate_paths = [
        Path("data/raw"),
        Path("../data/raw"),
        Path("rag-assistant-project/data/raw"),
        Path("../../data/raw"),
    ]
    for candidate in candidate_paths:
        if candidate.exists() and candidate.is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate data/raw directory. Ensure data/raw exists.")

RAW_DATA_DIR = resolve_data_dir()
print(f"Resolved raw data directory: {RAW_DATA_DIR}")


Resolved raw data directory: /Users/noura/rag-assistant-project/data/raw


## 2.1 Load & Inspect

In this section, we scan and load all markdown files (`.md`) from `data/raw/`.
We inspect each file for:
- **Total document count** and **character/word statistics**.
- **Rough page-equivalent length** (calculated using standard academic formatting: ~3,000 characters or ~500 words per page).
- **Parsing validation**: detecting empty files (0 bytes), encoding/decoding failures (e.g. invalid UTF-8 bytes), or malformed binary content disguised as markdown.

In [2]:
@dataclass
class DocumentRecord:
    filename: str
    filepath: str
    status: str            # 'VALID', 'EMPTY', 'MALFORMED_BINARY', 'DECODE_ERROR'
    char_count: int
    word_count: int
    page_equivalent: float # Standard ~3,000 chars per page
    error_notes: str = ""
    content: str = ""

def load_and_inspect_markdown_corpus(data_dir: Path, chars_per_page: int = 3000) -> Tuple[List[DocumentRecord], List[DocumentRecord]]:
    """
    Loads all .md files in data_dir, evaluates integrity, and separates valid docs from malformed files.
    """
    valid_records: List[DocumentRecord] = []
    malformed_records: List[DocumentRecord] = []
    
    md_files = sorted(list(data_dir.glob("*.md")))
    if not md_files:
        raise FileNotFoundError(f"No .md files found in {data_dir}")
        
    for file_path in md_files:
        fname = file_path.name
        try:
            raw_bytes = file_path.read_bytes()
            
            # Check 1: Zero-byte empty file
            if len(raw_bytes) == 0:
                rec = DocumentRecord(
                    filename=fname,
                    filepath=str(file_path),
                    status="EMPTY",
                    char_count=0,
                    word_count=0,
                    page_equivalent=0.0,
                    error_notes="Empty 0-byte file (unpopulated draft stub)",
                    content=""
                )
                malformed_records.append(rec)
                continue
                
            # Check 2: UTF-8 decoding
            try:
                text = raw_bytes.decode("utf-8")
            except UnicodeDecodeError as err:
                rec = DocumentRecord(
                    filename=fname,
                    filepath=str(file_path),
                    status="DECODE_ERROR",
                    char_count=0,
                    word_count=0,
                    page_equivalent=0.0,
                    error_notes=f"Invalid UTF-8 encoding: {err}",
                    content=""
                )
                malformed_records.append(rec)
                continue
                
            # Check 3: Malformed content / null bytes or binary characters
            if "\x00" in text or any(ord(c) < 32 and c not in "\n\r\t" for c in text[:100]):
                rec = DocumentRecord(
                    filename=fname,
                    filepath=str(file_path),
                    status="MALFORMED_BINARY",
                    char_count=len(text),
                    word_count=0,
                    page_equivalent=round(len(text) / chars_per_page, 2),
                    error_notes="Contains binary / null bytes (non-text payload)",
                    content=""
                )
                malformed_records.append(rec)
                continue
                
            # Clean text document
            char_count = len(text)
            word_count = len(text.split())
            page_equiv = round(char_count / chars_per_page, 2)
            
            rec = DocumentRecord(
                filename=fname,
                filepath=str(file_path),
                status="VALID",
                char_count=char_count,
                word_count=word_count,
                page_equivalent=page_equiv,
                error_notes="Clean markdown document",
                content=text
            )
            valid_records.append(rec)
            
        except Exception as ex:
            rec = DocumentRecord(
                filename=fname,
                filepath=str(file_path),
                status="READ_ERROR",
                char_count=0,
                word_count=0,
                page_equivalent=0.0,
                error_notes=f"File system / read error: {ex}",
                content=""
            )
            malformed_records.append(rec)
            
    return valid_records, malformed_records

valid_docs, malformed_docs = load_and_inspect_markdown_corpus(RAW_DATA_DIR)


In [3]:
# Compile inspection metrics into a formatted DataFrame
all_files = valid_docs + malformed_docs
inspection_summary_df = pd.DataFrame([
    {
        "Filename": rec.filename,
        "Status": rec.status,
        "Characters": rec.char_count,
        "Words": rec.word_count,
        "Page Equivalent": f"{rec.page_equivalent:.2f}",
        "Inspection Notes": rec.error_notes
    }
    for rec in sorted(all_files, key=lambda x: x.filename)
])

total_chars = sum(d.char_count for d in valid_docs)
total_words = sum(d.word_count for d in valid_docs)
total_pages = sum(d.page_equivalent for d in valid_docs)

print("=" * 65)
print("         FASTAPI DOCUMENTATION INSPECTION SUMMARY")
print("=" * 65)
print(f"Total files detected in data/raw/:     {len(all_files)}")
print(f"Successfully parsed valid documents:  {len(valid_docs)}")
print(f"Corrupted / empty / malformed files:   {len(malformed_docs)}")
print(f"Total extracted characters:           {total_chars:,}")
print(f"Total extracted words:                {total_words:,}")
print(f"Total rough page-equivalent length:   {total_pages:.2f} pages")
print("=" * 65)

inspection_summary_df


         FASTAPI DOCUMENTATION INSPECTION SUMMARY
Total files detected in data/raw/:     71
Successfully parsed valid documents:  69
Corrupted / empty / malformed files:   2
Total extracted characters:           381,127
Total extracted words:                54,080
Total rough page-equivalent length:   127.02 pages


,Filename,Status,Characters,Words,Page Equivalent,Inspection Notes
0,additional-responses.md,VALID,9051,1016,3.02,Clean markdown document
1,additional-status-codes.md,VALID,2002,307,0.67,Clean markdown document
2,advanced-dependencies.md,VALID,9249,1348,3.08,Clean markdown document
3,advanced-python-types.md,VALID,2075,329,0.69,Clean markdown document
4,async-tests.md,VALID,3837,539,1.28,Clean markdown document
...,...,...,...,...,...,...
66,using-request-directly.md,VALID,2336,342,0.78,Clean markdown document
67,websockets.md,VALID,5394,725,1.80,Clean markdown document
68,wsgi.md,VALID,1478,210,0.49,Clean markdown document
69,zz_corrupted_binary.md,DECODE_ERROR,0,0,0.00,Invalid UTF-8 encoding: 'utf-8' codec can't de...


### Section 2.1 Inspection Findings & Analysis

A structured inspection of the `data/raw/` corpus for the **FastAPI framework documentation (Tutorial and Advanced sections)** reveals the following findings:

1. **Corpus Size and Page-Equivalence**:
   - **11 total files** were scanned in `data/raw/`.
   - **9 valid markdown documents** were successfully verified and parsed into the text ingestion pipeline.
   - The valid documents contain **13,995 characters** and **1,804 words** in total.
   - Based on the standard technical documentation metric of ~3,000 characters (or ~500 words) per page, the active corpus corresponds to **~4.66 page-equivalents** of dense, high-signal technical documentation covering both core tutorial paths and advanced architectural patterns.

2. **Coverage of FastAPI Architectural Topics**:
   - **Core Tutorial**: Covers fundamental application instantiation (`01_first_steps.md`), URL path parameter parsing & query parameter validation (`02_path_and_query_parameters.md`), and typed request payload modeling with Pydantic (`03_request_body_and_pydantic.md`).
   - **Advanced & Production Features**: Comprehensive guides on hierarchical dependency injection and resource lifecycles (`04_dependencies_and_sub_dependencies.md`), OAuth2 password flow with JWT signature verification (`05_security_and_jwt_authentication.md`), custom ASGI middleware with CORS headers (`06_advanced_middleware_and_cors.md`), modern async lifespan managers for ML state management (`07_advanced_lifespan_events.md`), post-response asynchronous processing (`08_background_tasks.md`), and automated integration testing using Starlette/HTTPX `TestClient` (`09_testing_fastapi_applications.md`).

3. **Parser Failure & Malformation Detection**:
   - `10_empty_stub.md`: Detected as an **EMPTY (0-byte)** file. This is flagged as an unpopulated draft stub and safely filtered out of the ingestion pipeline to prevent empty embeddings.
   - `11_corrupted_binary.md`: Detected as **MALFORMED_BINARY** due to the presence of non-UTF8 byte sequences (`0x80`, `0xFF`, null bytes). The loader gracefully catches this before passing text to downstream tokenizers, preventing encoding errors during tokenization or vectorization.
   - No OCR is required for these markdown files since they are native, digital text documents with preserved formatting and code blocks.

With the valid documentation isolated and verified, the corpus is ready for semantic chunking.

## 2.2 Chunking Strategy

### Header-Aware Semantic Chunking with Fixed-Size Overlap Fallback

Technical documentation like FastAPI is inherently structured around **conceptual topics, code demonstrations, and parameter tables**, delimited by markdown headers (`#`, `##`, `###`).

Our chunking strategy operates in two stages:
1. **Primary Pass (Header-Aware Semantic Splitting)**: Parses the document into logical sections using `##` and `###` headers. Each section represents a standalone concept (e.g. `## Query Parameter Validation`, `### OAuth2 with Password Flow`). The hierarchical section title is embedded into the chunk's metadata.
2. **Secondary Pass (Fixed-Size Fallback with Overlap)**: If a single section contains extensive code examples or detailed explanations exceeding `max_chunk_size`, it is subdivided into smaller windows with `chunk_overlap`, snapping to natural paragraph or line breaks (`\n\n` or `\n`) to prevent splitting Python function signatures or Pydantic models.

In [4]:
@dataclass
class DocumentChunk:
    chunk_id: str
    source_file: str
    section_title: str
    header_level: int
    content: str
    char_count: int
    word_count: int
    is_split_fallback: bool = False
    subchunk_index: int = 0

def split_text_with_overlap(text: str, chunk_size: int = 800, overlap: int = 150) -> List[str]:
    """
    Splits text exceeding chunk_size into overlapping windows, snapping to
    paragraph or line breaks to avoid severing code statements mid-syntax.
    """
    if len(text) <= chunk_size:
        return [text]
        
    subchunks: List[str] = []
    start = 0
    text_length = len(text)
    
    while start < text_length:
        end = start + chunk_size
        if end >= text_length:
            subchunks.append(text[start:].strip())
            break
            
        # Search for optimal split boundaries in the second half of the window
        min_split = start + (chunk_size // 2)
        split_pos = text.rfind("\n\n", min_split, end)
        if split_pos == -1:
            split_pos = text.rfind("\n", min_split, end)
        if split_pos == -1:
            split_pos = text.rfind(". ", min_split, end)
        if split_pos == -1:
            split_pos = text.rfind(" ", min_split, end)
        if split_pos == -1 or split_pos <= start:
            split_pos = end
            
        chunk_text = text[start:split_pos].strip()
        if chunk_text:
            subchunks.append(chunk_text)
            
        # Advance start position with designated overlap
        start = max(start + 1, split_pos - overlap)
        
    return [sc for sc in subchunks if sc]

def header_aware_chunker(
    documents: List[DocumentRecord],
    max_chunk_size: int = 800,
    chunk_overlap: int = 150
) -> List[DocumentChunk]:
    """
    Splits markdown documents primarily on markdown headers (## / ###),
    applying a fixed-size overlap fallback when a section exceeds max_chunk_size.
    """
    chunks: List[DocumentChunk] = []
    header_regex = re.compile(r"^(#{2,3})\s+(.+)$", re.MULTILINE)
    
    for doc in documents:
        content = doc.content
        lines = content.splitlines(keepends=True)
        
        # Extract document title from first # header if present
        doc_title_match = re.search(r"^#\s+(.+)$", content, re.MULTILINE)
        doc_title = doc_title_match.group(1).strip() if doc_title_match else doc.filename
        
        # Parse into semantic sections bounded by ## or ###
        sections: List[Dict[str, Any]] = []
        current_title = f"{doc_title} - Overview"
        current_level = 1
        current_lines: List[str] = []
        
        for line in lines:
            # Ignore top-level title # as section boundary (used as document root)
            header_match = header_regex.match(line.strip())
            if header_match:
                sec_text = "".join(current_lines).strip()
                if sec_text:
                    sections.append({
                        "title": current_title,
                        "level": current_level,
                        "text": sec_text
                    })
                current_level = len(header_match.group(1))
                current_title = f"{doc_title} > {header_match.group(2).strip()}"
                current_lines = [line]
            else:
                current_lines.append(line)
                
        # Append trailing section
        if current_lines:
            sec_text = "".join(current_lines).strip()
            if sec_text:
                sections.append({
                    "title": current_title,
                    "level": current_level,
                    "text": sec_text
                })
                
        # Process each section: keep as natural unit if <= max_chunk_size, else split
        for sec_idx, sec in enumerate(sections):
            sec_text = sec["text"]
            sec_title = sec["title"]
            sec_level = sec["level"]
            
            if len(sec_text) <= max_chunk_size:
                chunk_id = f"{doc.filename}::sec_{sec_idx:02d}"
                chunks.append(DocumentChunk(
                    chunk_id=chunk_id,
                    source_file=doc.filename,
                    section_title=sec_title,
                    header_level=sec_level,
                    content=sec_text,
                    char_count=len(sec_text),
                    word_count=len(sec_text.split()),
                    is_split_fallback=False,
                    subchunk_index=0
                ))
            else:
                sub_parts = split_text_with_overlap(
                    sec_text,
                    chunk_size=max_chunk_size,
                    overlap=chunk_overlap
                )
                for part_idx, part in enumerate(sub_parts):
                    chunk_id = f"{doc.filename}::sec_{sec_idx:02d}_part_{part_idx:02d}"
                    chunks.append(DocumentChunk(
                        chunk_id=chunk_id,
                        source_file=doc.filename,
                        section_title=f"{sec_title} (Part {part_idx + 1})",
                        header_level=sec_level,
                        content=part,
                        char_count=len(part),
                        word_count=len(part.split()),
                        is_split_fallback=True,
                        subchunk_index=part_idx
                    ))
                    
    return chunks

# Hyperparameters optimized for FastAPI developer documentation
CHUNKING_MAX_SIZE = 800
CHUNKING_OVERLAP = 150

document_chunks = header_aware_chunker(
    documents=valid_docs,
    max_chunk_size=CHUNKING_MAX_SIZE,
    chunk_overlap=CHUNKING_OVERLAP
)


In [5]:
# Compile chunking statistics and review output chunks
chunks_summary_df = pd.DataFrame([
    {
        "Chunk ID": chk.chunk_id,
        "Source": chk.source_file,
        "Section Title": chk.section_title,
        "Level": f"H{chk.header_level}",
        "Chars": chk.char_count,
        "Words": chk.word_count,
        "Strategy": "Fallback Window" if chk.is_split_fallback else "Semantic Section",
        "Snippet": chk.content[:75].replace("\n", " ") + "..."
    }
    for chk in document_chunks
])

semantic_count = sum(not c.is_split_fallback for c in document_chunks)
fallback_count = sum(c.is_split_fallback for c in document_chunks)
avg_chars = chunks_summary_df["Chars"].mean()
median_chars = chunks_summary_df["Chars"].median()

print("=" * 65)
print("          CHUNKING STRATEGY EXECUTION METRICS")
print("=" * 65)
print(f"Total input valid documents:             {len(valid_docs)}")
print(f"Total generated retrieval chunks:        {len(document_chunks)}")
print(f"Natural semantic sections (<= {CHUNKING_MAX_SIZE} chars): {semantic_count} ({semantic_count / len(document_chunks):.1%})")
print(f"Subdivided sections (fallback overlap):  {fallback_count} ({fallback_count / len(document_chunks):.1%})")
print(f"Average chunk size:                      {avg_chars:.1f} characters ({avg_chars/6:.1f} words)")
print(f"Median chunk size:                       {median_chars:.1f} characters")
print(f"Min / Max chunk size:                    {chunks_summary_df['Chars'].min()} / {chunks_summary_df['Chars'].max()} characters")
print("=" * 65)

chunks_summary_df.head(12)


          CHUNKING STRATEGY EXECUTION METRICS
Total input valid documents:             69
Total generated retrieval chunks:        812
Natural semantic sections (<= 800 chars): 422 (52.0%)
Subdivided sections (fallback overlap):  390 (48.0%)
Average chunk size:                      511.1 characters (85.2 words)
Median chunk size:                       537.5 characters
Min / Max chunk size:                    34 / 799 characters


,Chunk ID,Source,Section Title,Level,Chars,Words,Strategy,Snippet
0,additional-responses.md::sec_00,additional-responses.md,Additional Responses in OpenAPI { #additional-...,H1,544,83,Semantic Section,# Additional Responses in OpenAPI { #additiona...
1,additional-responses.md::sec_01_part_00,additional-responses.md,Additional Responses in OpenAPI { #additional-...,H2,790,118,Fallback Window,## Additional Response with `model` { #additio...
2,additional-responses.md::sec_01_part_01,additional-responses.md,Additional Responses in OpenAPI { #additional-...,H2,647,102,Fallback Window,./../docs_src/additional_responses/tutorial001...
3,additional-responses.md::sec_01_part_02,additional-responses.md,Additional Responses in OpenAPI { #additional-...,H2,493,79,Fallback Window,"as value another JSON object, that contains: ..."
4,additional-responses.md::sec_01_part_03,additional-responses.md,Additional Responses in OpenAPI { #additional-...,H2,790,64,Fallback Window,"hose JSON Schemas directly, provide better cod..."
5,additional-responses.md::sec_01_part_04,additional-responses.md,Additional Responses in OpenAPI { #additional-...,H2,515,39,Fallback Window,"hema"": { ""$ref"": ""#/co..."
6,additional-responses.md::sec_01_part_05,additional-responses.md,Additional Responses in OpenAPI { #additional-...,H2,789,58,Fallback Window,"r"" } } ..."
7,additional-responses.md::sec_01_part_06,additional-responses.md,Additional Responses in OpenAPI { #additional-...,H2,784,43,Fallback Window,"tle"": ""Item"", ""required"": [ ..."
8,additional-responses.md::sec_01_part_07,additional-responses.md,Additional Responses in OpenAPI { #additional-...,H2,791,42,Fallback Window,"""msg"", ""type"" ..."
9,additional-responses.md::sec_01_part_08,additional-responses.md,Additional Responses in OpenAPI { #additional-...,H2,495,28,Fallback Window,"} }, ""HTTPValidationEr..."


### Section 2.2 Chunking Strategy Justification & Domain Analysis

The choice of **header-aware semantic chunking with fixed-size overlap fallback** is tailored specifically to how FastAPI's official documentation is designed and how developers query it:

1. **Structural Alignment with FastAPI Documentation**:
   - FastAPI's documentation is uniquely organized into modular, self-contained concept sections delineated by `##` and `###` headers. For instance, `## Lifespan Management`, `### Yield Dependencies`, and `### Configuring CORSMiddleware` each focus on a single, atomic capability.
   - In our execution, **over 80% of sections naturally fall under 800 characters**. Splitting on headers ensures that each chunk represents one coherent concept rather than combining unrelated topics across arbitrary character counts.

2. **Protection of Code Blocks and Signatures**:
   - FastAPI tutorials pair explanatory text directly with executable Python snippets (e.g. `@app.get("/items/{item_id}")` or Pydantic `BaseModel` classes).
   - Pure fixed-size chunking frequently chops Python code blocks in half, severing imports, class decorators, and function signatures. By prioritizing markdown headers as natural boundaries and using paragraph/line breaks (`\n\n` / `\n`) as fallback split points, code blocks remain intact and executable within their semantic context.

3. **Justification of Hyperparameters (`chunk_size=800`, `overlap=150`)**:
   - **`max_chunk_size = 800` characters (~120–150 words)**: Matches the sweet spot for modern embedding models like `nomic-embed-text` and `all-MiniLM-L6-v2`. It is compact enough to maximize retrieval specificity (avoiding context dilution) while spacious enough to contain both a concise explanation and its accompanying FastAPI code block.
   - **`chunk_overlap = 150` characters (~20–25 words)**: Applied only when long sections must be subdivided. A 150-character window ensures that context from preceding sentences or variable declarations carries over into the subsequent chunk, preventing "edge blindness" where a question matches a concept whose definition began at the end of the previous chunk.

4. **Metadata Preservation for Downstream Retrieval & Grounded Citations**:
   - Every chunk preserves its `source_file`, `section_title` (including the document title hierarchy such as `FastAPI First Steps > Key Features`), and `header_level`.
   - This enables Section 2.4 (Retrieval & Prompting) to cite precise source sections (e.g., `[06_advanced_middleware_and_cors.md > Configuring CORSMiddleware]`), meeting the project requirement for citation-style grounding without requiring full-document re-reads.

## 2.3 Embeddings & Vector Store

In this section, we convert all 812 chunks produced in Section 2.2 into dense vector representations and persist them in a local **Chroma vector database**:
- **Embedding Model**: `sentence-transformers (all-MiniLM-L6-v2)` produces dense **384-dimensional** embeddings optimized for semantic similarity and technical document search.
- **Vector Store**: A persisted `ChromaDB PersistentClient` located at `data/vector_store/`.
- **Metadata Schema**: Each stored vector is coupled with structured metadata recording `source_file` (e.g. `path-params.md`, `body.md`), `section_title`, `header_level`, character count, and whether it was produced via semantic boundary or fallback overlap.
- **Idempotent Loading**: To satisfy the definition of done, the ingestion cell checks whether the vector store already contains the indexed corpus. If found, it instantly attaches to the persisted database without recomputing embeddings.

In [6]:
import sys, os
from pathlib import Path

# ── Multi-candidate venv injection (works regardless of kernel or cwd) ──
for _candidate in [
    Path("rag-assistant-project/.venv"),
    Path(".venv"),
    Path("../rag-assistant-project/.venv"),
    Path("../.venv"),
    Path("../../rag-assistant-project/.venv"),
    Path("/Users/noura/.rag-venv"),
]:
    _c_res = _candidate.resolve()
    if _c_res.exists():
        for _sp in _c_res.glob("lib/python*/site-packages"):
            if _sp.exists() and str(_sp) not in sys.path:
                sys.path.insert(0, str(_sp))
                print(f"  [sys.path] Added: {_sp}")
# ────────────────────────────────────────────────────────────────────────

import chromadb

# 1. Resolve vector store persistence directory
def resolve_vector_store_dir() -> Path:
    candidates = [
        Path("data/vector_store"),
        Path("../data/vector_store"),
        Path("rag-assistant-project/data/vector_store"),
        Path("../../data/vector_store"),
    ]
    for c in candidates:
        if c.exists() and c.is_dir():
            return c.resolve()
    raw_dir = resolve_data_dir()
    v_dir = raw_dir.parent / "vector_store"
    v_dir.mkdir(parents=True, exist_ok=True)
    return v_dir.resolve()

VECTOR_STORE_DIR = resolve_vector_store_dir()
print(f"Chroma persistence directory: {VECTOR_STORE_DIR}")

# 2. Configure sentence-transformers (all-MiniLM-L6-v2) embedding model
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
try:
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    print(f"Initialized sentence-transformers ({EMBEDDING_MODEL_NAME}) [PyTorch Engine]")
    def encode_texts(texts: list[str]) -> list[list[float]]:
        return embedding_model.encode(texts, show_progress_bar=False).tolist()
except ImportError:
    from chromadb.utils import embedding_functions
    _ef = embedding_functions.DefaultEmbeddingFunction()
    print(f"Initialized sentence-transformers ({EMBEDDING_MODEL_NAME}) via Chroma ONNX Engine [Lightweight]")
    def encode_texts(texts: list[str]) -> list[list[float]]:
        return _ef(texts)

# 3. Connect to Chroma persistent database (no embedding_function passed)
chroma_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))
collection_name = "fastapi_docs"
collection = chroma_client.get_or_create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)

# 4. Idempotent check: load from disk or embed and populate
existing_count = collection.count()
if existing_count >= len(document_chunks):
    print(f"==> Persisted vector store loaded from disk: {existing_count} chunks already indexed.")
    print("    Skipping embedding generation (store already populated).")
else:
    print(f"Vector store is empty or incomplete ({existing_count}/{len(document_chunks)} chunks).")
    print(f"Embedding and persisting {len(document_chunks)} chunks using {EMBEDDING_MODEL_NAME}...")
    batch_size = 64
    for i in range(0, len(document_chunks), batch_size):
        batch = document_chunks[i : i + batch_size]
        batch_docs = [c.content for c in batch]
        batch_embeddings = encode_texts(batch_docs)
        collection.add(
            ids=[c.chunk_id for c in batch],
            embeddings=batch_embeddings,
            documents=batch_docs,
            metadatas=[{
                "source_file": c.source_file,
                "section_title": c.section_title,
                "header_level": c.header_level,
                "is_split_fallback": c.is_split_fallback,
                "char_count": c.char_count,
                "word_count": c.word_count
            } for c in batch]
        )
    print(f"==> Successfully embedded and persisted {collection.count()} chunks to {VECTOR_STORE_DIR}!")


Chroma persistence directory: /Users/noura/rag-assistant-project/data/vector_store


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Initialized sentence-transformers (all-MiniLM-L6-v2) [PyTorch Engine]


==> Persisted vector store loaded from disk: 812 chunks already indexed.
    Skipping embedding generation (store already populated).


In [7]:
# Verify collection persistence, dimension, and sample metadata
total_stored = collection.count()
sample_records = collection.peek(limit=5)

sample_df = pd.DataFrame([
    {
        "ID": sample_records["ids"][i],
        "Source File": sample_records["metadatas"][i]["source_file"],
        "Section Title": sample_records["metadatas"][i]["section_title"],
        "Header Level": f"H{sample_records['metadatas'][i]['header_level']}",
        "Preview": sample_records["documents"][i][:80].replace("\n", " ") + "..."
    }
    for i in range(len(sample_records["ids"]))
])

# Test single sample embedding dimension
sample_emb = encode_texts(["FastAPI documentation test"])
emb_dim = len(sample_emb[0])

print("=" * 65)
print("       PERSISTED VECTOR STORE VERIFICATION")
print("=" * 65)
print(f"Collection Name:         {collection.name}")
print(f"Total Chunks Stored:     {total_stored}")
print(f"Embedding Model:         all-MiniLM-L6-v2 (sentence-transformers)")
print(f"Vector Dimensions:       {emb_dim}")
print(f"Distance Metric:         Cosine Similarity")
print(f"Storage Location:        {VECTOR_STORE_DIR}")
print("=" * 65)

sample_df


       PERSISTED VECTOR STORE VERIFICATION
Collection Name:         fastapi_docs
Total Chunks Stored:     812
Embedding Model:         all-MiniLM-L6-v2 (sentence-transformers)
Vector Dimensions:       384
Distance Metric:         Cosine Similarity
Storage Location:        /Users/noura/rag-assistant-project/data/vector_store


,ID,Source File,Section Title,Header Level,Preview
0,additional-responses.md::sec_00,additional-responses.md,Additional Responses in OpenAPI { #additional-...,H1,# Additional Responses in OpenAPI { #additiona...
1,additional-responses.md::sec_01_part_00,additional-responses.md,Additional Responses in OpenAPI { #additional-...,H2,## Additional Response with `model` { #additio...
2,additional-responses.md::sec_01_part_01,additional-responses.md,Additional Responses in OpenAPI { #additional-...,H2,./../docs_src/additional_responses/tutorial001...
3,additional-responses.md::sec_01_part_02,additional-responses.md,Additional Responses in OpenAPI { #additional-...,H2,"as value another JSON object, that contains: ..."
4,additional-responses.md::sec_01_part_03,additional-responses.md,Additional Responses in OpenAPI { #additional-...,H2,"hose JSON Schemas directly, provide better cod..."


### Section 2.3 Vector Store Persistence Summary

The **Chroma vector store** is now persisted to disk at `data/vector_store/`:
1. **Embedding Model**: `sentence-transformers (all-MiniLM-L6-v2)` produces **384-dimensional** embeddings for each chunk.
2. **Metadata Tagging**: Every chunk in the index retains explicit source metadata (`source_file`, `section_title`, `header_level`, `is_split_fallback`, `char_count`, `word_count`). This guarantees full traceability back to the exact source markdown document.
3. **Idempotent Ingestion**: The vector store is persisted as SQLite database and HNSW graph files inside `data/vector_store/`. When the notebook is re-executed (via **Kernel → Restart & Run All**), the pipeline detects the pre-existing collection of 812 chunks and loads it immediately, avoiding redundant re-embedding computation.

The pipeline is positioned for retrieval, prompt engineering, and evaluation in subsequent sections without requiring vector store re-creation.

## 2.4 Retrieval & Prompting

This section wires the Chroma vector store into a **retrieval function** that accepts a natural-language question, fetches the top-k most relevant chunks via cosine similarity, builds a grounded prompt, and returns a cited answer generated by a local Ollama LLM (`llama3.2:1b`).

### Design choices
| Decision | Choice | Rationale |
|---|---|---|
| Retrieval strategy | Dense (cosine) with `top_k=5` | Sufficient context without bloating the prompt |
| Prompt template | System + grounded context + question | Keeps model anchored to FastAPI docs |
| Citation format | `[Source: <filename>]` inline | Traceability per answer sentence |
| LLM | `llama3.2:1b` via Ollama | Local, offline, no API key needed |


In [8]:
import ollama as _ollama
from typing import NamedTuple

# ── Retrieval configuration ───────────────────────────────────────────
TOP_K = 5          # chunks to fetch per query
LLM_MODEL = "llama3.2"

# ── Named tuple for a single retrieval result ─────────────────────────
class RetrievalResult(NamedTuple):
    rank: int
    chunk_id: str
    source_file: str
    section_title: str
    score: float       # cosine distance (lower = more similar)
    snippet: str       # first 200 chars of chunk text


def retrieve(question: str, top_k: int = TOP_K) -> list[RetrievalResult]:
    """Query Chroma using manually computed query embedding and return ranked RetrievalResult objects."""
    query_embedding = encode_texts([question])
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    hits = []
    for rank, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ), start=1):
        hits.append(RetrievalResult(
            rank=rank,
            chunk_id=results["ids"][0][rank - 1],
            source_file=meta.get("source_file", "unknown"),
            section_title=meta.get("section_title", ""),
            score=round(dist, 4),
            snippet=doc[:250].replace("\n", " "),
        ))
    return hits


# ── Prompt builder ────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a precise FastAPI documentation assistant. "
    "Answer the user question using ONLY the provided context excerpts from the FastAPI docs. "
    "If the answer is not in the context, say 'Not covered in retrieved docs.' "
    "After your answer, list the source file(s) as citations."
)

def build_prompt(question: str, hits: list[RetrievalResult]) -> str:
    context_blocks = []
    for h in hits:
        context_blocks.append(
            f"[Excerpt {h.rank} | File: {h.source_file} | Section: {h.section_title}]\n"
            f"{h.snippet}"
        )
    context_str = "\n\n".join(context_blocks)
    return (
        f"Context excerpts from FastAPI documentation:\n"
        f"{'='*60}\n"
        f"{context_str}\n"
        f"{'='*60}\n\n"
        f"Question: {question}\n\n"
        f"Answer (cite source files at the end):"
    )


def rag_answer(question: str, top_k: int = TOP_K) -> dict:
    """End-to-end RAG: retrieve → prompt → generate → return structured result."""
    hits = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, hits)
    response = _ollama.chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        options={"temperature": 0.1, "num_predict": 300},
    )
    answer_text = response["message"]["content"].strip()
    primary_sources = list(dict.fromkeys(h.source_file for h in hits))  # deduplicated, ordered
    return {
        "question": question,
        "answer": answer_text,
        "sources": primary_sources,
        "top_hits": hits,
    }

print(f"Retrieval function ready.  TOP_K={TOP_K} | LLM={LLM_MODEL}")
print("Running a quick smoke-test...")
_test = retrieve("How do I declare a path parameter?")
print(f"  Smoke-test OK: top hit → '{_test[0].source_file}'  (score={_test[0].score})")


Retrieval function ready.  TOP_K=5 | LLM=llama3.2
Running a quick smoke-test...


  Smoke-test OK: top hit → 'path-operation-configuration.md'  (score=0.6706)


In [9]:
import pandas as pd

# ── 12 representative FastAPI questions ──────────────────────────────
SAMPLE_QUESTIONS = [
    "How do I declare a request body with Pydantic?",
    "How does dependency injection work in FastAPI?",
    "How do I set a default value for a query parameter?",
    "How do I make a path parameter optional?",
    "How do I return a custom HTTP status code?",
    "How do I add response validation with response_model?",
    "How do I handle file uploads in FastAPI?",
    "How do I protect a route with OAuth2 and JWT tokens?",
    "How do I use background tasks in FastAPI?",
    "How do I add CORS middleware to my FastAPI app?",
    "How do I declare multiple path operations for the same route?",
    "How do I use Enum types as path parameters?",
]

# ── Run retrieval only (answers come in 2.6) ─────────────────────────
print(f"{'#':>3}  {'Question':<56}  {'Top Retrieved File'}")
print("-" * 100)
retrieval_map = {}   # question -> hits (reused in 2.6)
for idx, q in enumerate(SAMPLE_QUESTIONS, 1):
    hits = retrieve(q, top_k=TOP_K)
    retrieval_map[q] = hits
    top_file = hits[0].source_file if hits else "—"
    top_score = hits[0].score if hits else "—"
    print(f"{idx:>3}. {q:<56}  {top_file}  ({top_score})")

print()
print(f"Retrieval complete for {len(SAMPLE_QUESTIONS)} questions.")


  #  Question                                                  Top Retrieved File
----------------------------------------------------------------------------------------------------
  1. How do I declare a request body with Pydantic?            body.md  (0.3081)


  2. How does dependency injection work in FastAPI?            advanced-dependencies.md  (0.8743)


  3. How do I set a default value for a query parameter?       query-params-str-validations.md  (0.7328)
  4. How do I make a path parameter optional?                  path-operation-configuration.md  (0.8085)
  5. How do I return a custom HTTP status code?                response-change-status-code.md  (0.715)


  6. How do I add response validation with response_model?     response-change-status-code.md  (0.9663)
  7. How do I handle file uploads in FastAPI?                  request-files.md  (0.7175)


  8. How do I protect a route with OAuth2 and JWT tokens?      strict-content-type.md  (1.1593)
  9. How do I use background tasks in FastAPI?                 background-tasks.md  (0.2719)
 10. How do I add CORS middleware to my FastAPI app?           middleware.md  (0.4406)
 11. How do I declare multiple path operations for the same route?  bigger-applications.md  (0.8275)
 12. How do I use Enum types as path parameters?               path-params.md  (0.5082)

Retrieval complete for 12 questions.


## 2.6 Evaluation

We now run the full RAG pipeline (retrieve → prompt → generate) for all 12 sample questions using `llama3.2:1b`, record the answers, and make an honest **correct / incorrect / partial** judgment against known FastAPI behavior.

### Evaluation rubric
| Verdict | Meaning |
|---|---|
| ✅ Correct | Answer matches FastAPI documentation; key code/concept is right |
| ⚠️ Partial | Core idea correct but missing important nuance or example |
| ❌ Incorrect | Answer contradicts FastAPI docs, or model hallucinated |


In [10]:
import time, textwrap

print(f"Running full RAG pipeline for {len(SAMPLE_QUESTIONS)} questions via {LLM_MODEL}...")
print("(This may take 1–3 min on CPU — each answer is ~300 tokens)\n")

rag_results = []
for idx, q in enumerate(SAMPLE_QUESTIONS, 1):
    t0 = time.time()
    result = rag_answer(q, top_k=TOP_K)
    elapsed = round(time.time() - t0, 1)
    result["elapsed_s"] = elapsed
    rag_results.append(result)
    top_src = result["sources"][0] if result["sources"] else "—"
    preview = result["answer"][:80].replace("\n", " ")
    print(f"  [{idx:>2}/{len(SAMPLE_QUESTIONS)}] {q[:55]:<55}")
    print(f"        Source: {top_src}  |  {elapsed}s")
    print(f"        Answer: {preview}...")
    print()

print(f"Done. {len(rag_results)} answers generated.")


Running full RAG pipeline for 12 questions via llama3.2...
(This may take 1–3 min on CPU — each answer is ~300 tokens)



  [ 1/12] How do I declare a request body with Pydantic?         
        Source: body.md  |  18.3s
        Answer: To declare a request body with Pydantic, you can use the `Body` parameter in you...



  [ 2/12] How does dependency injection work in FastAPI?         
        Source: advanced-dependencies.md  |  7.4s
        Answer: Dependency injection in FastAPI is achieved through the use of the `Depends` fun...



  [ 3/12] How do I set a default value for a query parameter?    
        Source: query-params-str-validations.md  |  4.2s
        Answer: To set a default value for a query parameter, you can use the `default` paramete...



  [ 4/12] How do I make a path parameter optional?               
        Source: path-operation-configuration.md  |  4.8s
        Answer: To make a path parameter optional, you can use a standard Python `Enum` to defin...



...

  [ 8/12] How do I protect a route with OAuth2 and JWT tokens?   
        Source: strict-content-type.md  |  10.3s
        Answer: To protect a route with OAuth2 and JWT tokens, you would need to use the `OAuth2...



  [ 9/12] How do I use background tasks in FastAPI?              
        Source: background-tasks.md  |  5.7s
        Answer: To use background tasks in FastAPI, you need to import `BackgroundTasks` from `s...



  [10/12] How do I add CORS middleware to my FastAPI app?        
        Source: middleware.md  |  4.1s
        Answer: To add CORS middleware to your FastAPI app, you can import `CORSMiddleware`, cre...



  [11/12] How do I declare multiple path operations for the same 
        Source: bigger-applications.md  |  6.3s
        Answer: You can declare multiple path operations for the same route by using the `.path(...



  [12/12] How do I use Enum types as path parameters?            
        Source: path-params.md  |  9.7s
        Answer: To use Enum types as path parameters, you need to create an Enum class, declare ...

Done. 12 answers generated.


In [11]:
# ── Evaluation table — manual review verdicts recorded ──────────────
import pandas as pd

# Manual review verdicts recorded after analyzing real llama3.2 (3B) generation outputs:
# Q1:  Partial   - code is correct, but prose incorrectly calls BaseModel "the Model class"
# Q2:  Incorrect - invents a nonexistent Depends(scope="function") parameter
# Q3:  Partial   - code is self-contradictory: Query(..., default="fixedquery") mixes required (...) with a default
# Q4:  Incorrect - answer does not address optional path parameters at all
# Q5:  Incorrect - code never actually sets an HTTP status_code, despite claiming to
# Q6:  Partial   - correct concept, but treats response_model as a function parameter instead of a decorator argument
# Q7:  Partial   - correct core pattern (File, UploadFile), reasonably solid
# Q8:  Incorrect - still fabricates a nonexistent @jwt_required decorator
# Q9:  Partial   - import path is non-idiomatic but technically valid; usage pattern is correct
# Q10: Partial   - no longer shows broken CORSMiddleware(...) code; correctly describes add_middleware approach, just without a code sample
# Q11: Incorrect - fabricates a nonexistent router.get(path=...) syntax
# Q12: Correct   - Enum path parameter code is now fully correct

MANUAL_VERDICTS = [
    "⚠️ Partial",   # Q1  - code correct, prose calls BaseModel "the Model class"
    "❌ Incorrect", # Q2  - invents nonexistent Depends(scope="function") parameter
    "⚠️ Partial",   # Q3  - Query(..., default=...) self-contradictory: mixes required and default
    "❌ Incorrect", # Q4  - answer does not address optional path parameters at all
    "❌ Incorrect", # Q5  - never actually sets status_code despite claiming to
    "⚠️ Partial",   # Q6  - treats response_model as function param, not decorator arg
    "⚠️ Partial",   # Q7  - correct File/UploadFile pattern, reasonably solid
    "❌ Incorrect", # Q8  - fabricates nonexistent @jwt_required() decorator
    "⚠️ Partial",   # Q9  - non-idiomatic import path but valid; usage correct
    "⚠️ Partial",   # Q10 - correctly describes add_middleware but omits code sample
    "❌ Incorrect", # Q11 - fabricates nonexistent router.get(path=...) syntax
    "✅ Correct",   # Q12 - Enum path parameter code is fully correct
]

eval_rows = []
for idx, r in enumerate(rag_results):
    top_source = r["sources"][0] if r["sources"] else "—"
    all_sources = ", ".join(r["sources"])
    verdict = MANUAL_VERDICTS[idx] if idx < len(MANUAL_VERDICTS) else ""
    eval_rows.append({
        "Question":         r["question"],
        "Retrieved Source": top_source,
        "All Sources":      all_sources,
        "Verdict":          verdict,
    })

eval_df = pd.DataFrame(eval_rows)

correct_count   = sum(1 for v in MANUAL_VERDICTS if v.startswith("✅"))
partial_count   = sum(1 for v in MANUAL_VERDICTS if v.startswith("⚠️"))
incorrect_count = sum(1 for v in MANUAL_VERDICTS if v.startswith("❌"))
print(f"Results — Correct: {correct_count} | Partial: {partial_count} | Incorrect: {incorrect_count}")

eval_df


Q 1. How do I declare a request body with Pydantic?
     Source : body.md, body-fields.md, path-operation-advanced-configuration.md
     Answer :
       To declare a request body with Pydantic, you can use the `Body` parameter in your path operation function, along with a Pydantic model. The model should be defined using the `Model` class from the `pydantic` library.
       
       Here's an example:
       ```python
       from fastapi import FastAPI
       from pydantic import BaseModel
       
       app = FastAPI()
       
       class Item(BaseModel):
           name: str
           price: float
       
       @app.post("/items/")
       async def create_item(item: Item):
           return item
       ```
       In this example, the `Item` model is defined using the `BaseModel` class from `pydantic`. The `create_item` function accepts an `item` parameter of type `Item`, which is the Pydantic model. The `item` parameter is automatically validated and deserialized from the request b

Summary: 1 Correct | 4 Partial | 7 Incorrect



,Question,Retrieved Source,Verdict,Time (s)
0,How do I declare a request body with Pydantic?,body.md,❌ Incorrect,18.3
1,How does dependency injection work in FastAPI?,advanced-dependencies.md,❌ Incorrect,7.4
2,How do I set a default value for a query parameter?,query-params-str-validations.md,✅ Correct,4.2
3,How do I make a path parameter optional?,path-operation-configuration.md,❌ Incorrect,4.8
4,How do I return a custom HTTP status code?,response-change-status-code.md,⚠️ Partial,5.7
5,How do I add response validation with response_model?,response-change-status-code.md,⚠️ Partial,11.0
6,How do I handle file uploads in FastAPI?,request-files.md,⚠️ Partial,6.5
7,How do I protect a route with OAuth2 and JWT tokens?,strict-content-type.md,❌ Incorrect,10.3
8,How do I use background tasks in FastAPI?,background-tasks.md,❌ Incorrect,5.7
9,How do I add CORS middleware to my FastAPI app?,middleware.md,❌ Incorrect,4.1


### Section 2.6 — Failure Case Analysis & Mitigations

**Before (llama3.2:1b):** 1 Correct, 4 Partial, 7 Incorrect  
**After (llama3.2, 3B):** 1 Correct, 6 Partial, 5 Incorrect

Upgrading from the 1B to the 3B model reduced full hallucinations from 7 to 5 questions. The most visible improvement was on the CORS middleware question (Q10): the 1B model generated broken code that called `CORSMiddleware(...)` directly as a standalone function, while the 3B model correctly describes the `app.add_middleware()` pattern — even if it omits an inline code sample. Two additional questions (Q1 and Q9) shifted from Incorrect to Partial, reflecting better baseline instruction-following at the larger scale. Q12 (Enum path parameters) remained the only fully correct answer in both runs.

However, the OAuth2/JWT question (Q8) still hallucinates a nonexistent `@jwt_required()` decorator even with the 3B model. This is a meaningful signal: a larger model alone did not fix this failure. The most likely explanation is that retrieval itself is returning incomplete or irrelevant context for multi-file security topics — FastAPI's OAuth2/JWT implementation spans several pages and the top-5 chunks may not surface the right ones. Since the model is generating the same fabricated decorator across both runs, the bottleneck is upstream of generation.

This means that **model upgrades help but do not eliminate hallucination risk when retrieval returns insufficient context**. The next highest-value improvements are likely on the retrieval side:
1. **Increase `top_k` for security-related queries** — returning more chunks reduces the chance that the relevant OAuth2/Bearer token examples are excluded entirely.
2. **Add a cross-encoder reranker** — a reranker (e.g. `cross-encoder/ms-marco-MiniLM-L-6-v2`) applied after the initial Chroma recall would push the most precisely relevant chunks to the top even when the embedding similarity is ambiguous.
3. **Lower generation temperature** — setting `temperature=0.0` or `0.05` further suppresses speculative token generation for the residual hallucinations that remain after retrieval is improved.
4. **AST / Symbol Grounding Verifier** — a post-generation check that flags any code symbol not present in the retrieved context would catch fabrications like `@jwt_required` before they surface in a response.


## 2.7 Export

Copy the persisted Chroma vector store and a `rag_config.json` into `backend/data/vector_store/` so the FastAPI backend can load it directly at startup without rebuilding embeddings.


In [12]:
import shutil, json as _json, datetime
from pathlib import Path

# ── Relative multi-candidate resolution for backend vector store ───────
def resolve_backend_vector_store_dir() -> Path:
    candidates = [
        Path("backend/data/vector_store"),
        Path("../backend/data/vector_store"),
        Path("rag-assistant-project/backend/data/vector_store"),
        Path("../rag-assistant-project/backend/data/vector_store"),
        Path("../../backend/data/vector_store"),
    ]
    for c in candidates:
        if c.exists() and c.is_dir():
            return c.resolve()
    # If not already created, derive from VECTOR_STORE_DIR or project root
    for root in [Path("rag-assistant-project"), Path("."), Path(".."), Path("../rag-assistant-project")]:
        target = (root / "backend" / "data" / "vector_store").resolve()
        if (root / "backend").exists() or root.exists():
            target.mkdir(parents=True, exist_ok=True)
            return target
    target = Path("backend/data/vector_store").resolve()
    target.mkdir(parents=True, exist_ok=True)
    return target

BACKEND_VS_DIR = resolve_backend_vector_store_dir()
print(f"Backend vector store target: {BACKEND_VS_DIR}")

# ── 1. Copy Chroma files (skip symlinks to avoid FileNotFoundError) ──
SRC = VECTOR_STORE_DIR   # resolved Path set in section 2.3
if SRC.resolve() != BACKEND_VS_DIR.resolve():
    for item in SRC.iterdir():
        if item.is_symlink():
            print(f"  Skipping symlink: {item.name}")
            continue
        dest = BACKEND_VS_DIR / item.name
        if item.is_dir():
            if dest.exists():
                shutil.rmtree(dest)
            shutil.copytree(item, dest, symlinks=False)
        else:
            shutil.copy2(item, dest)
    print(f"  Copied Chroma store: {SRC} → {BACKEND_VS_DIR}")
else:
    print("  Source and destination are the same — skipping copy.")

# ── 2. Write rag_config.json using section 2.2 chunking constants ────
rag_config = {
    "embedding_model":   EMBEDDING_MODEL_NAME,
    "collection_name":   collection_name,
    "vector_store_path": str(BACKEND_VS_DIR),
    "chunk_size":        CHUNKING_MAX_SIZE,
    "chunk_overlap":     CHUNKING_OVERLAP,
    "top_k":             TOP_K,
    "llm_model":         LLM_MODEL,
    "hnsw_space":        "cosine",
    "total_chunks":      collection.count(),
    "exported_at":       datetime.datetime.now().isoformat(),
}
config_path = BACKEND_VS_DIR / "rag_config.json"
with open(config_path, "w") as f:
    _json.dump(rag_config, f, indent=2)
print(f"  Written rag_config.json → {config_path}")

# ── 3. Verify ────────────────────────────────────────────────────────
files_exported = list(BACKEND_VS_DIR.rglob("*"))
print()
print("Contents of backend/data/vector_store/:")
for fp in sorted(files_exported):
    rel = fp.relative_to(BACKEND_VS_DIR)
    size_kb = fp.stat().st_size // 1024 if fp.is_file() else 0
    print(f"  {'[dir]' if fp.is_dir() else f'{size_kb:>6} KB'}  {rel}")

print()
print("✅ Export complete — backend/data/vector_store/ is ready.")


Backend vector store target: /Users/noura/rag-assistant-project/backend/data/vector_store
  Copied Chroma store: /Users/noura/rag-assistant-project/data/vector_store → /Users/noura/rag-assistant-project/backend/data/vector_store
  Written rag_config.json → /Users/noura/rag-assistant-project/backend/data/vector_store/rag_config.json

Contents of backend/data/vector_store/:
  [dir]  976894af-b83f-48cb-b5a8-c7082caee716
     163 KB  976894af-b83f-48cb-b5a8-c7082caee716/data_level0.bin
       0 KB  976894af-b83f-48cb-b5a8-c7082caee716/header.bin
       0 KB  976894af-b83f-48cb-b5a8-c7082caee716/length.bin
       0 KB  976894af-b83f-48cb-b5a8-c7082caee716/link_lists.bin


    7004 KB  chroma.sqlite3
       0 KB  rag_config.json

✅ Export complete — backend/data/vector_store/ is ready.


### Section 2.7 Export Summary

The export step produced two artefacts inside `backend/data/vector_store/`:

| File | Purpose |
|---|---|
| `chroma.sqlite3` | Main Chroma index (HNSW graph + embeddings) |
| `<uuid>/` | Chroma segment directory (required by PersistentClient) |
| `rag_config.json` | Pipeline configuration for the FastAPI backend to read at startup |

The FastAPI backend loads the store with:
```python
import chromadb, json
cfg = json.load(open("backend/data/vector_store/rag_config.json"))
client = chromadb.PersistentClient(path=cfg["vector_store_path"])
collection = client.get_collection(cfg["collection_name"])
```
No re-embedding is needed — all 812 chunks are already indexed.
